# 8.4 · 梯度提升推导 / Gradient Boosting Derivation

> **课程定位 / Where this fits**
> 第 4 课，**Part 8 · 集成学习**。
> Lesson 4, **Part 8 · Ensemble Learning**.
>
> 8.3 的 AdaBoost 靠"重加权样本"纠错，只能配指数损失。**梯度提升(GBDT)** 把 boosting 推广到**任意可导损失**：核心洞察是把"加一棵树"看成**在函数空间做梯度下降**——每棵树去拟合损失的**负梯度（伪残差）**。5.8/4.13 用过 GBDT，这一课**把数学推导讲透**：为什么残差就是负梯度、不同损失对应不同伪残差。这是理解 XGBoost(8.5) 的地基。
> AdaBoost (8.3) corrects errors by reweighting samples and only fits the exponential loss. **Gradient boosting (GBDT)** generalizes boosting to **any differentiable loss**: the key insight is to view "adding a tree" as **gradient descent in function space** — each tree fits the loss's **negative gradient (pseudo-residual)**. GBDT was used in 5.8/4.13; this lesson **derives the math fully**: why residuals are the negative gradient, and how different losses give different pseudo-residuals. The foundation for XGBoost (8.5).
>
> 💼 **实战/面试视角**："梯度提升的'梯度'是对什么求导 / 伪残差怎么来 / 不同损失" 是 boosting 进阶必考。
> 💼 **Practical/interview angle:** "what is the 'gradient' / where do pseudo-residuals come from / different losses" — advanced boosting.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $F_m(\mathbf{x})$ —— 第 $m$ 步的加法模型输出 / additive model after step $m$
> - $L(y, F)$ —— 损失函数 / loss function
> - $r_{im}$ —— 伪残差 = 损失对 $F$ 的负梯度 / pseudo-residual = negative gradient w.r.t. $F$
> - $\nu$ —— 学习率/收缩 / learning rate (shrinkage)

> 💡 **面试相关 / Interview-relevant**
> - "梯度提升的'梯度'是对预测值还是参数求导"（出镜率 ★★★★★，对预测值/函数）
> - "为什么平方损失的伪残差就是普通残差"（★★★★★）
> - "不同损失的伪残差（平方/对数/绝对）"（★★★★）
> - "学习率(shrinkage)的作用"（★★★★★）
> - "GBDT vs AdaBoost"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解"加一棵树 = 函数空间梯度下降"。
   Understand "adding a tree = gradient descent in function space".
2. 推出**伪残差 = 损失对 F 的负梯度**。
   Derive pseudo-residual = negative gradient of the loss w.r.t. F.
3. **从零**实现通用 GBDT（可换损失）。
   Implement a generic GBDT from scratch (swappable loss).
4. 看不同损失（平方/对数/绝对）对应的伪残差。
   See pseudo-residuals for squared/log/absolute losses.
5. 理解学习率(shrinkage)的作用。
   Understand the role of learning rate (shrinkage).

## 目录 / TOC
1. [先建直觉：函数空间的梯度下降 ⭐](#1)
2. [伪残差 = 负梯度（推导）⭐](#2)
3. [📈 数据 + 从零通用 GBDT ⭐](#3)
4. [不同损失的伪残差 ⭐](#4)
5. [学习率 shrinkage + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：函数空间的梯度下降 ⭐ / Intuition: Gradient Descent in Function Space

普通梯度下降优化**参数**：$\theta \leftarrow \theta - \eta\,\nabla_\theta L$，沿损失对参数的负梯度走一小步。
Ordinary gradient descent optimizes **parameters**: $\theta \leftarrow \theta - \eta\,\nabla_\theta L$, stepping along the negative gradient w.r.t. parameters.

梯度提升的天才之处：**把整个预测函数 $F$ 本身当成"要优化的变量"**。我们想让损失 $L(y, F)$ 变小，那就让 $F$ 沿**损失对 $F$ 的负梯度**方向走一步。但 $F$ 是个函数、不是一组数——所以我们**训一棵树去"拟合"这个负梯度方向**，再把树加进 $F$。这就是"**函数空间的梯度下降**"：
The genius of gradient boosting: **treat the prediction function $F$ itself as the variable to optimize**. To decrease $L(y, F)$, step $F$ along the **negative gradient of the loss w.r.t. $F$**. But $F$ is a function, not numbers — so we **train a tree to "fit" that negative-gradient direction** and add it to $F$. That's "**gradient descent in function space**":

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu\, h_m(\mathbf{x}), \quad \text{其中 } h_m \text{ 拟合负梯度}$$

**面试要点**："梯度提升的梯度是对什么求导？"——答案是**对预测值 $F(\mathbf{x})$**（不是对参数），这是它和普通梯度下降的关键区别。
**Interview point:** "What is the gradient taken with respect to?" — w.r.t. the **prediction $F(\mathbf{x})$** (not parameters), the key difference from ordinary gradient descent.


<a id="2"></a>
## 2. 伪残差 = 负梯度（推导）⭐ / Pseudo-residual = Negative Gradient

第 $m$ 步，每个样本的**伪残差**定义为损失对当前预测的负梯度：
At step $m$, each sample's **pseudo-residual** is the negative gradient of the loss w.r.t. the current prediction:

$$r_{im} = -\left[\frac{\partial L(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right]_{F=F_{m-1}}$$

然后训一棵树 $h_m$ **拟合这些 $r_{im}$**，把它加进模型。关键来了——**对平方损失，伪残差恰好就是普通残差**（面试高频"为什么"）：
Then train a tree $h_m$ to **fit these $r_{im}$** and add it. The key — **for squared loss, the pseudo-residual is exactly the ordinary residual** (a frequent "why"):

$$L = \tfrac12(y-F)^2 \;\Rightarrow\; \frac{\partial L}{\partial F} = -(y-F) \;\Rightarrow\; r = -\frac{\partial L}{\partial F} = y - F \;(\text{即残差!})$$

所以 4.13/5.8 里"每棵树拟合残差"只是**平方损失下的特例**——一般情形是"拟合负梯度"。下表给出常用损失的伪残差：
So "each tree fits the residual" (4.13/5.8) is just the **squared-loss special case** — in general it's "fit the negative gradient". The table gives pseudo-residuals for common losses:

| 损失 loss | 伪残差 $r_i$ | 解读 |
|---|---|---|
| **平方 squared** $\frac12(y-F)^2$ | $y_i - F_i$ | 普通残差 |
| **绝对 absolute** $\lvert y-F\rvert$ | $\text{sign}(y_i - F_i)$ | 只看方向（抗异常）|
| **对数 log loss**（分类）| $y_i - \sigma(F_i)$ | 真实减预测概率 = $(y-p)$ |


<a id="3"></a>
## 3. 数据 + 从零通用 GBDT ⭐ / Generic GBDT from Scratch

实现一个**损失可换**的 GBDT：只要给它一个"算伪残差"的函数，它就能优化任意损失。下面在合成回归数据上跑，先用平方损失，对照 sklearn。
We implement a **loss-swappable** GBDT: give it a "pseudo-residual" function and it optimizes any loss. We run it on synthetic regression data with squared loss and compare to sklearn.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 合成回归数据 / synthetic regression
x = np.sort(rng.uniform(-3, 3, 400))
y = x**3 - 2*x + rng.normal(0, 2, 400)
X = x.reshape(-1, 1)

# 通用 GBDT: 传入一个 "负梯度(伪残差)" 函数即可优化对应损失 / generic GBDT
def gbdt_fit(X, y, neg_gradient, n_trees=100, lr=0.1, max_depth=3, init=None):
    F = np.full(len(y), y.mean() if init is None else init)   # 初始预测
    trees = []
    for _ in range(n_trees):
        r = neg_gradient(y, F)                  # 伪残差 = 损失对 F 的负梯度
        tree = DecisionTreeRegressor(max_depth=max_depth).fit(X, r)   # 树拟合伪残差
        F += lr * tree.predict(X)               # 沿负梯度方向走一步(步长 lr)
        trees.append(tree)
    return trees, (y.mean() if init is None else init)

def predict(X, trees, F0, lr=0.1):
    return F0 + lr * sum(t.predict(X) for t in trees)

# 平方损失: 负梯度 = y - F (普通残差) / squared loss → residual
trees, F0 = gbdt_fit(X, y, neg_gradient=lambda y, F: y - F, n_trees=100, lr=0.1)
my_mse = mean_squared_error(y, predict(X, trees, F0))

from sklearn.ensemble import GradientBoostingRegressor
sk = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y)
print(f"从零 GBDT(平方损失) 训练 MSE = {my_mse:.3f}")
print(f"sklearn GBDT      训练 MSE = {mean_squared_error(y, sk.predict(X)):.3f}")
print("→ 一致; 核心就是'伪残差 = y-F, 树拟合它'这一行(平方损失特例)")


<a id="4"></a>
## 4. 不同损失的伪残差 ⭐ / Pseudo-residuals for Different Losses

通用框架的威力：**只换"负梯度函数"，就能优化不同损失**，无需改其它代码。下面在**含异常值**的数据上对比"平方损失"和"绝对损失"的 GBDT——绝对损失的伪残差是 $\text{sign}(y-F)$（只看方向、不看大小），所以**对异常值稳健**（异常点的巨大残差不会主导，因为只贡献 ±1）。
The power of the generic framework: **just swap the "negative-gradient function"** to optimize a different loss, no other code changes. On data **with outliers**, we compare squared-loss vs absolute-loss GBDT — the absolute-loss pseudo-residual is $\text{sign}(y-F)$ (direction only, ignoring magnitude), making it **robust to outliers** (a huge residual contributes only ±1, so it can't dominate).


In [ ]:
# 注入异常值 / inject outliers
y_out = y.copy()
out_idx = rng.choice(len(y), 20, replace=False)
y_out[out_idx] += rng.normal(0, 40, 20)          # 20 个大异常 / 20 big outliers

# 平方损失 vs 绝对损失 (只换负梯度函数!) / swap the negative-gradient function
trees_sq, F0_sq = gbdt_fit(X, y_out, lambda y, F: y - F, n_trees=100, lr=0.1)            # 残差
trees_ab, F0_ab = gbdt_fit(X, y_out, lambda y, F: np.sign(y - F), n_trees=100, lr=0.1,
                           init=np.median(y_out))   # sign(残差); 绝对损失初值用中位数

x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y_out, alpha=0.2, s=10, label="数据(含异常)")
ax.scatter(x[out_idx], y_out[out_idx], c="red", s=20, label="异常值")
ax.plot(x_plot, predict(x_plot, trees_sq, F0_sq), "C1", lw=2, label="GBDT 平方损失(被异常拉偏)")
ax.plot(x_plot, predict(x_plot, trees_ab, F0_ab), "C2", lw=2, label="GBDT 绝对损失(稳健)")
ax.set_ylim(-35, 35); ax.legend(fontsize=8); ax.set_title("换损失=换伪残差: 绝对损失(sign)对异常值更稳健")
plt.tight_layout(); plt.show()
print("绝对损失的伪残差=sign(y-F), 异常点的大残差只贡献 ±1 → 拉不动模型 → 稳健")
print("→ 这就是通用 GBDT 的威力: 只换负梯度函数就适配不同任务/鲁棒性(分位数/Huber 同理)")


<a id="5"></a>
## 5. 学习率 shrinkage + 小结 ⭐ / Shrinkage & Summary

每加一棵树乘一个**学习率 $\nu$（shrinkage）**：$F_m = F_{m-1} + \nu\,h_m$。它让每步只迈一**小步**，是 boosting 最重要的正则手段之一。
Each tree is scaled by a **learning rate $\nu$ (shrinkage)**: $F_m = F_{m-1} + \nu\,h_m$. It makes each step **small**, one of boosting's most important regularizers.

- **$\nu$ 小**（0.01-0.1）：每步保守、泛化更好，但**需要更多树**才收敛。
  **Small $\nu$** (0.01-0.1): conservative steps, better generalization, but **needs more trees** to converge.
- **$\nu$ 大**：快但易过拟合。
  **Large $\nu$**: fast but overfit-prone.
- 经验配方：**小学习率 + 多树 + 早停**（GBDT 树太多会过拟合，与 RF 相反）。
  Recipe: **small LR + many trees + early stopping** (too many trees overfit GBDT, unlike RF).


In [ ]:
# 学习率 × 树数: 小 lr 需更多树 / learning rate vs number of trees
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for lr in [0.03, 0.1, 0.5]:
    mses = []
    ns = [5, 10, 20, 40, 80, 150]
    for nt in ns:
        trees, F0 = gbdt_fit(X, y, lambda y, F: y - F, n_trees=nt, lr=lr)
        mses.append(mean_squared_error(y, predict(X, trees, F0, lr)))
    ax.plot(ns, mses, "o-", label=f"lr={lr}")
ax.set_xlabel("树数 n_trees"); ax.set_ylabel("训练 MSE"); ax.legend()
ax.set_title("学习率×树数: 小 lr 每步保守, 需更多树才降到同等水平")
plt.tight_layout(); plt.show()
print("小 lr 需更多树收敛但泛化更好; 大 lr 快但易过拟合 → 小 lr+多树+早停 是标准配方")


```
梯度提升 = 函数空间的梯度下降: 把预测函数 F 当变量, 每棵树拟合损失对 F 的负梯度
梯度对'预测值 F'求导(不是对参数) — 这是面试核心区别点
伪残差 r = -∂L/∂F:
  平方损失 → r=y-F(普通残差, 这是 4.13/5.8 的特例)
  绝对损失 → r=sign(y-F)(只看方向, 抗异常)
  对数损失 → r=y-σ(F)=y-p
通用 GBDT: 只换"负梯度函数"就适配任意可导损失/鲁棒性
学习率 ν(shrinkage): 小步更稳需更多树; 小lr+多树+早停 是标准配方
GBDT(拟合负梯度, 任意损失) vs AdaBoost(重加权, 固定指数损失)
```

### 💡 面试速查 / Interview cheat-sheet
1. **梯度提升 = 函数空间梯度下降**; 梯度对**预测值 F** 求导(不是参数)。
   Gradient boosting = gradient descent in function space; gradient w.r.t. the prediction F (not parameters).
2. **伪残差 = -∂L/∂F**; 平方损失下就是普通残差 y-F。
   Pseudo-residual = -∂L/∂F; under squared loss it's the ordinary residual y-F.
3. **换损失=换伪残差**: 绝对→sign(抗异常), 对数→y-p。
   Swap the loss = swap the pseudo-residual: absolute→sign (robust), log→y-p.
4. **学习率小+树多+早停**; GBDT 树太多会过拟合(与 RF 相反)。
   Small LR + many trees + early stopping; too many trees overfit GBDT (unlike RF).
5. **GBDT(拟合负梯度) vs AdaBoost(重加权)**; GBDT 更通用。
   GBDT (fits the negative gradient) vs AdaBoost (reweighting); GBDT is more general.

### 下一节 / Next
**8.5 XGBoost/LightGBM/CatBoost 对比**——三大现代 boosting 库在同一数据上横向比较: 实现差异(直方图/leaf-wise/有序提升)、速度、精度、何时用哪个。
**8.5 XGBoost/LightGBM/CatBoost** — the three modern boosting libraries compared on the same data: implementation differences (histogram/leaf-wise/ordered boosting), speed, accuracy, and when to use which.
